# Boussinesq — competitive baselines using their own original prediction protocols

**This is the intended comparison, not the previous K=30 architectural-adaptation notebook.**

**F-FNO**: one-step autoregressive (100 predictions from u0). **U-Net**: one-step autoregressive. **CNO**: directly predicts the field at queried physical time, (u0,t) -> u(t), no rollout. **U-NO-H10**: 10 observed history fields -> next field, 91 recursive predictions. **AWFNO** remains K=30 (its own windowed formulation).

Implemented following the project's Burgers and Cahn–Hilliard baselines and annular geometry adaptations. The original 70 test trajectories and weighted physical annular L2 are unchanged. **U-NO receives 10 observed test fields**, so its full L2 is deliberately not reported; it only has final-time and post-t10 L2 and must be discussed separately from u0-only methods.

## Significant computational cost

The one-step and direct-time baseline training sets now have **100 samples per training trajectory**, rather than just three K30 windows. The completed **N_train=25** experiments are preserved as the original 200-epoch, fixed-small-batch reference. This notebook now defaults to **N_train=50,100,200 only** (N=400 excluded). On a 90 GiB RTX PRO 6000, it benchmarks multiple GPU-safe batch sizes and chooses the best measured training throughput per architecture. It permits early stopping after validation plateaus while retaining 200 as the maximum epoch count. Since batch sizes and early stopping differ from N25, the two groups of results represent **different optimization settings**; the CSV labels the configuration and should not be interpreted as a fully matched learning-curve sweep. To make all sizes training-protocol matched, rerun N25 under the updated settings later. The original and K30 results are not modified.

**Expected runtime:** without acceleration, N50+100+200 across four models takes roughly 18 h extrapolated from measured N25; autotuning may reduce this, but the actual reduction is unverified. Training logs print seconds/epoch and a maximum-time estimate every 10 epochs. The experiment records actual chosen batch and optimizer updates. The original 70-test-trajectory evaluation is unchanged.

This Colab uses readable Python scripts from GitHub, independently saves and resumes checkpoints on Drive, streams training logs, and first runs model-specific architecture shape/gradient checks.

In [6]:
#@title 1. Drive / GPU / maintained UNO dependency
from google.colab import drive
from pathlib import Path
import os, sys, time, hashlib, urllib.request, subprocess
import torch
drive.mount('/content/gdrive')
assert torch.cuda.is_available(), 'Enable a GPU via Runtime > Change runtime type.'
DATASET=Path('/content/gdrive/MyDrive/anus/hyteg_annulus_c_grid_64x128_samples500.pt')
OUTPUT_ROOT=Path('/content/gdrive/MyDrive/anus/Boussinesq_JCP')
NATIVE_ROOT=OUTPUT_ROOT/'native_reference_2d'
assert DATASET.is_file(), f'Dataset unavailable: {DATASET}'
NATIVE_ROOT.mkdir(parents=True,exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install','-q','neuraloperator==2.0.0'],check=True)
print('GPU:',torch.cuda.get_device_name(0))
print('Data GiB:',round(DATASET.stat().st_size/1024**3,3))
print('New outputs:',NATIVE_ROOT)

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Data GiB: 3.098
New outputs: /content/gdrive/MyDrive/anus/Boussinesq_JCP/native_reference_2d


In [7]:
#@title 2. Configure remaining N=50,100,200 experiments (N=25 preserved)
MODELS='CNO,UNet,UNO' #@param {type:'string'}
SAMPLE_SIZES='50,100,200' #@param {type:'string'}
SEEDS='1234' #@param {type:'string'}
EPOCHS=200 #@param {type:'integer'}
AUTOTUNE_BATCH=True #@param {type:'boolean'}
GPU_MEMORY_FRACTION=0.75 #@param {type:'number'}
AUTOTUNE_STEPS=2 #@param {type:'integer'}
EARLY_STOP_PATIENCE=40 #@param {type:'integer'}
EARLY_STOP_MIN_EPOCHS=80 #@param {type:'integer'}
EARLY_STOP_REL_DELTA=0.001 #@param {type:'number'}
SMOKE_TEST=True #@param {type:'boolean'}
FORCE_RETRAIN=False #@param {type:'boolean'}
SAVE_FULL_TEST_PREDICTIONS=False #@param {type:'boolean'}
AUTO_DISCONNECT=False #@param {type:'boolean'}
assert set(MODELS.split(',')).issubset({'FFNO','CNO','UNet','UNO'})
assert all(int(k) in (25,50,100,200) for k in SAMPLE_SIZES.split(','))
assert all(int(k)<=200 for k in SAMPLE_SIZES.split(','))
assert 0.1<GPU_MEMORY_FRACTION<=0.95
print('Selected models:',MODELS,'training sizes:',SAMPLE_SIZES,
      'max epochs:',EPOCHS,'autobatch:',AUTOTUNE_BATCH)
print('Early stop:',EARLY_STOP_PATIENCE,'epochs without >',
      EARLY_STOP_REL_DELTA*100,'% relative validation improvement, after',
      EARLY_STOP_MIN_EPOCHS,'epochs')
print('NOTE: Existing N25 fixed-batch results remain, but new optimizers use tuned batches.')

Selected models: CNO,UNet,UNO training sizes: 50,100,200 max epochs: 200 autobatch: True
Early stop: 40 epochs without > 0.1 % relative validation improvement, after 80 epochs
NOTE: Existing N25 fixed-batch results remain, but new optimizers use tuned batches.


In [8]:
#@title 3. Download latest readable source with no-cache checks
GITHUB='https://raw.githubusercontent.com/dimigourz/AWFNO/main/Boussinesq2D/'
SOURCE_NAMES=['annulus2d_jcp_experiments.py',
              'reference_architectures_2d.py',
              'native_method_baselines_2d.py']
for filename in SOURCE_NAMES:
    request=urllib.request.Request(GITHUB+filename+'?refresh='+str(time.time_ns()),
                                    headers={'Cache-Control':'no-cache','Pragma':'no-cache'})
    with urllib.request.urlopen(request,timeout=90) as response:
        content=response.read()
    local=Path('/content')/filename
    local.write_bytes(content)
    print(filename,len(content),'bytes SHA256',hashlib.sha256(content).hexdigest())
DRIVER=Path('/content/native_method_baselines_2d.py')
assert DRIVER.is_file()
print('Source:',GITHUB)

annulus2d_jcp_experiments.py 103262 bytes SHA256 a4d78c78bda2703e408ae90ed501dd56334294c9082c086ab2516df6a7acc883
reference_architectures_2d.py 15591 bytes SHA256 fb5eb6e417e02441cacec154a4451a63a725d78e5863353ef0a8ac100ecdd4a5
native_method_baselines_2d.py 34223 bytes SHA256 4f0b923402239e95cce610fec65d29f6da4a8ab74e47ed18a9ba701a471651ec
Source: https://raw.githubusercontent.com/dimigourz/AWFNO/main/Boussinesq2D/


In [9]:
#@title 4. Native-method smoke tests (per architecture)
# Refresh the native architectures even when Cell 3 was not rerun.
# This avoids using the stale U-Net constructor (width instead of channels).
import urllib.request, ast, time
source_url=GITHUB+'reference_architectures_2d.py?refresh='+str(time.time_ns())
req=urllib.request.Request(source_url,headers={'Cache-Control':'no-cache','Pragma':'no-cache'})
with urllib.request.urlopen(req,timeout=90) as response:
    reference_bytes=response.read()
reference_path=Path('/content/reference_architectures_2d.py')
reference_path.write_bytes(reference_bytes)
reference_ast=ast.parse(reference_bytes.decode('utf-8'),filename=str(reference_path))
native_factory=next(
    (node for node in reference_ast.body
     if isinstance(node,ast.FunctionDef) and node.name=='make_native_model'),None)
if native_factory is None:
    raise RuntimeError('The refreshed architecture module has no native model factory.')
unet_calls=[node for node in ast.walk(native_factory)
            if isinstance(node,ast.Call) and isinstance(node.func,ast.Name)
            and node.func.id=='UNet2D']
if len(unet_calls)!=1 or any(k.arg=='width' for k in unet_calls[0].keywords):
    raise RuntimeError('Stale U-Net source: native factory must call UNet2D(..., channels=32), not width=32.')
if not any(k.arg=='channels' and isinstance(k.value,ast.Constant) and k.value.value==32
           for k in unet_calls[0].keywords):
    raise RuntimeError('U-Net constructor mismatch; refusing to run training.')
print('[SOURCE] Refreshed native model architectures; verified UNet2D(..., channels=32).',flush=True)

ENV=os.environ.copy()
ENV.update({
  'DATA_PATH':str(DATASET),'OUTPUT_ROOT':str(OUTPUT_ROOT),
  'NATIVE_OUTPUT':str(NATIVE_ROOT),'NATIVE_MODELS':MODELS,
  'NATIVE_SAMPLE_SIZES':SAMPLE_SIZES,'NATIVE_SEEDS':SEEDS,
  'NATIVE_EPOCHS':str(EPOCHS),
  'NATIVE_AUTOTUNE_BATCH':'1' if AUTOTUNE_BATCH else '0',
  'NATIVE_MAX_GPU_FRACTION':str(GPU_MEMORY_FRACTION),
  'NATIVE_AUTOTUNE_STEPS':str(AUTOTUNE_STEPS),
  'NATIVE_EARLY_STOP_PATIENCE':str(EARLY_STOP_PATIENCE),
  'NATIVE_EARLY_STOP_MIN_EPOCHS':str(EARLY_STOP_MIN_EPOCHS),
  'NATIVE_EARLY_STOP_REL_DELTA':str(EARLY_STOP_REL_DELTA),
  'NATIVE_FORCE_RETRAIN':'1' if FORCE_RETRAIN else '0',
  'NATIVE_SAVE_ROLLOUTS':'1' if SAVE_FULL_TEST_PREDICTIONS else '0',
  'PYTHONUNBUFFERED':'1',
  'SPLIT_MODE':'contiguous','TIME_SUBSAMPLE':'1',
  'N_VAL':'30','N_TEST':'70',
})
if SMOKE_TEST:
    for model_name in MODELS.split(','):
        print('\n[SMOKE] ',model_name,flush=True)
        env=dict(ENV);env['NATIVE_SMOKE_ONLY']='1';env['NATIVE_MODELS']=model_name
        path=NATIVE_ROOT/f'smoke_{model_name}.log'
        with path.open('w') as output:
            with subprocess.Popen([sys.executable,'-u',str(DRIVER)],
                    env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                    bufsize=1,text=True,errors='replace') as proc:
                for line in proc.stdout:
                    print(line,end='',flush=True);output.write(line);output.flush()
                status=proc.wait()
        if status:raise RuntimeError(f'Smoke test failed for {model_name}: {path}')
        print('[SMOKE] PASS',model_name,flush=True)
else:print('Smoke tests disabled.')

[SOURCE] Refreshed native model architectures; verified UNet2D(..., channels=32).

[SMOKE]  CNO
Using device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
[SMOKE] Native protocols ('CNO',) on annulus grid
[SMOKE] PASS CNO, physical prediction shape=(2, 64, 128), parameters=1,624,753
[SMOKE] PASS CNO

[SMOKE]  UNet
Using device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
[SMOKE] Native protocols ('UNet',) on annulus grid
[SMOKE] PASS UNet, physical prediction shape=(2, 64, 128), parameters=7,766,273
[SMOKE] PASS UNet

[SMOKE]  UNO
Using device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
[SMOKE] Native protocols ('UNO',) on annulus grid
fno_skip='linear'
channel_mlp_skip='linear'
fno_skip='linear'
channel_mlp_skip='linear'
fno_skip='linear'
channel_mlp_skip='linear'
fno_skip='linear'
channel_mlp_skip='linear'
fno_skip='linear'
channel_mlp_skip='linear'
[SMOKE] PASS UNO, physical prediction shape=(2, 64, 128), parameters=3,179,329
[SMOKE] PASS UNO


In [10]:
#@title 5. Train the native protocols (logs and checkpoints saved to Drive)
from threading import Thread,Event
logfile=NATIVE_ROOT/'native_training_stdout.log'
stopped=Event()
def heartbeat():
    while not stopped.wait(180):
        print('[HEARTBEAT] Native comparison subprocess alive; last epoch prints show progress.',flush=True)
watcher=Thread(target=heartbeat,daemon=True);watcher.start()
try:
    with logfile.open('a',encoding='utf-8') as log:
        with subprocess.Popen([sys.executable,'-u',str(DRIVER)],
                env=ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                bufsize=1,text=True,errors='replace') as proc:
            for line in proc.stdout:
                print(line,end='',flush=True);log.write(line);log.flush()
            status=proc.wait()
        if status:
            raise RuntimeError(f'Native protocol run exited code {status}. See {logfile}')
finally:
    stopped.set();watcher.join()
print('Completed selected native protocols.',flush=True)

Using device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition

Loaded annulus data
-------------------
file: /content/gdrive/MyDrive/anus/hyteg_annulus_c_grid_64x128_samples500.pt
U                       : shape=(500, 64, 128, 101), dtype=float32, memory=1.541 GB
r shape: (64,)
theta shape: (128,)
t shape: (101,)
U min/max: -0.13059653341770172 1.0094236135482788

Geometry
--------
N, Nr, Ntheta, Nt: 500 64 128 101
r range          : 0.5 1.0
theta range      : 0.0 6.234097957611084
DR               : 0.007936507936507936
DTHETA           : 0.04908738521234052
DT_SNAPSHOT      : 1.0

Data split
----------
split mode       : contiguous
U_train_pool     : (400, 64, 128, 101)
U_val            : (30, 64, 128, 101)
U_test           : (70, 64, 128, 101)
train ids first  : [0 1 2 3 4]
val ids first    : [400 401 402 403 404]
test ids first   : [430 431 432 433 434]

Exact hash overlap check train vs validation
------------------------------------------
duplicates: 0

Exact hash overlap 

In [11]:
#@title 6. Read native comparison results
import pandas as pd
from IPython.display import display,Image
p=NATIVE_ROOT/'results.csv'
if p.is_file():
    df=pd.read_csv(p)
    names=['model','N_train','seed','protocol','observed_true_frames',
           'parameters','full_l2_mean','tail_from_t10_l2_mean','final_l2_mean',
           'training_seconds','effective_batch_size','epochs_completed',
           'early_stopped','optimizer_steps','training_protocol_version']
    display(df[[c for c in names if c in df.columns]].sort_values(['N_train','model']).reset_index(drop=True))
    print('NOTE: N25 was trained with the original fixed batch, unlike the autotuned N50–200.')
    print('Do not compare UNO_H10 full L2 against u0-only methods. Its full L2 is undefined.')
else:
    print('No completed model yet; examine the preceding output or',NATIVE_ROOT)
print('Outputs:',NATIVE_ROOT)

,model,N_train,seed,protocol,observed_true_frames,parameters,full_l2_mean,tail_from_t10_l2_mean,final_l2_mean,training_seconds,effective_batch_size,epochs_completed,early_stopped,optimizer_steps,training_protocol_version
0,CNO,25,1234,direct_time_u0_t,1,1624753,1.585041,1.578737,0.189088,1268.940061,NaN,NaN,NaN,62600,v1_original_batches
1,FFNO,25,1234,autoregressive_H1_to_H1,1,941826,1.701082,1.699306,0.226935,2089.290562,NaN,NaN,NaN,62600,v1_original_batches
2,UNO,25,1234,autoregressive_H10_to_H1,10,3179329,NaN,3.023635,0.470808,1033.329444,NaN,NaN,NaN,28600,v1_original_batches
3,UNet,25,1234,autoregressive_H1_to_H1,1,7766273,1.745607,1.743560,0.246820,216.023903,NaN,NaN,NaN,15800,v1_original_batches
4,CNO,50,1234,direct_time_u0_t,1,1624753,1.502623,1.497654,0.181360,1195.863668,32.0,130.0,True,20410,boussinesq_native_methods_v2_autobatch_earlystop
5,UNO,50,1234,autoregressive_H10_to_H1,10,3179329,NaN,2.978896,0.445725,978.275874,32.0,100.0,True,14300,boussinesq_native_methods_v2_autobatch_earlystop
6,UNet,50,1234,autoregressive_H1_to_H1,1,7766273,1.565947,1.565133,0.238427,396.945997,64.0,200.0,True,15800,boussinesq_native_methods_v2_autobatch_earlystop
7,CNO,100,1234,direct_time_u0_t,1,1624753,1.292860,1.290568,0.169761,2008.822503,32.0,110.0,True,34430,boussinesq_native_methods_v2_autobatch_earlystop
8,UNO,100,1234,autoregressive_H10_to_H1,10,3179329,NaN,2.856076,0.406019,2328.223301,32.0,120.0,True,34200,boussinesq_native_methods_v2_autobatch_earlystop
9,UNet,100,1234,autoregressive_H1_to_H1,1,7766273,1.441708,1.440731,0.208364,406.451807,32.0,100.0,True,31300,boussinesq_native_methods_v2_autobatch_earlystop


NOTE: N25 was trained with the original fixed batch, unlike the autotuned N50–200.
Do not compare UNO_H10 full L2 against u0-only methods. Its full L2 is undefined.
Outputs: /content/gdrive/MyDrive/anus/Boussinesq_JCP/native_reference_2d


In [12]:
#@title 7. Release GPU only after completion
if AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()
else:
    print('GPU retained. Disconnect manually after inspecting your numerical results.')

GPU retained. Disconnect manually after inspecting your numerical results.
